# Analiza datasetu
W tym notebooku przeprowadzona jest analiza zbioru danych, i tego jak wiele danych słów się pojawia i testowane są różne podejścia od tagowania istniejących rzadkich słów po inne metody.

## Uruchomienie w Google Colab

1. Wgraj do sesji katalog projektu (np. przez **Files** → *Upload* lub `git clone` repozytorium), tak aby obok notebooka były `hdf5_manager.py` i plik HDF5 (np. `./data/texts_eng.h5`).
2. Uruchom poniższą komórkę — doinstaluje zależności i model spaCy `en_core_web_sm`.
3. Ścieżki `./data/...` działają, gdy katalog roboczy to root projektu (w Colab: `%cd` do folderu z notebookiem).


In [6]:
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


if _in_colab():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "contractions",
            "pandas",
            "plotly",
            "h5py",
            "spacy",
            "tqdm",
            "huggingface_hub",
            "transformers",
            "sentencepiece",
            "nbformat",
        ]
    )
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    print("Colab: zależności i en_core_web_sm gotowe.")


## Ładowanie danych

In [7]:
from hdf5_manager import HDF5Loader

FILE = "./data/texts_eng.h5"

hdf5 = HDF5Loader(FILE)
data = hdf5.load()
hdf5_flags = hdf5.load_flags()
hdf5.close()
print(f"Loaded {len(data)} entries.")

Loaded 39817 entries.


## Standaryzacja tekstów

In [8]:
import contractions
import pandas as pd
import plotly.express as px

In [9]:
df = pd.DataFrame({"key": list(data.keys()), "text": list(data.values())})

df["word_count"] = df["text"].str.split().str.len()
df["source_flag"] = df["key"].map(hdf5_flags)
print(df.shape)
print(f"source_flag: {df['source_flag'].value_counts().to_string()}")
df.head()

(39817, 4)
source_flag: source_flag
deepl      39749
manual        58
invalid       10


,key,text,word_count,source_flag
0,001_000_M,There is a frog in a glass jar.,8,deepl
1,001_001_M,The dog looks inside.,4,deepl
2,001_002_M,The child lies on the bed and supports his hea...,13,deepl
3,001_003_M,"I said, the dog looks down at the frog sitting...",13,deepl
4,001_004_M,"In the second picture, the girl is sleeping un...",11,deepl


### Ładowanie metadanych (speaker_id)

Metadane z `data/pjm_data.csv` (pobierany z `croco-corp/pjm-data`, plik `splits/pjm_data.csv`) zawierają `speaker_id`, `video_type`, `task_id`, `sex`, `age`.  
Potrzebne do: deduplikacji per migacz, grupowego splitu train/val/test, stratyfikacji.

In [ ]:
import os
import shutil

from huggingface_hub import hf_hub_download

if not os.path.exists("./data/pjm_data.csv"):
    cached = hf_hub_download(
        repo_id="croco-corp/pjm-data", repo_type="dataset", filename="splits/pjm_data.csv"
    )
    os.makedirs("./data", exist_ok=True)
    shutil.copy(cached, "./data/pjm_data.csv")
    print("Downloaded pjm_data.csv")
else:
    print("pjm_data.csv already exists")

In [11]:
df_meta = pd.read_csv("./data/pjm_data.csv")

df["video_id"] = df["key"].str.split("_").str[0].astype(int)
_side_map = {"L": "left", "R": "right", "M": "-"}
df["speaker_side"] = df["key"].str.split("_").str[2].map(_side_map)

n_before = len(df)
df = df.merge(
    df_meta[["video_id", "speaker_side", "speaker_id", "sex", "age", "task_id", "video_type", "special_case"]],
    on=["video_id", "speaker_side"],
    how="left",
)
assert len(df) == n_before, f"Merge zmienił liczbę wierszy: {n_before} -> {len(df)}"

df["interaction_type"] = df["video_type"].str.split(";").str[0]

print(f"Unikalni migacze: {df['speaker_id'].nunique()}")
print(f"Wiersze bez speaker_id: {df['speaker_id'].isna().sum()}")
print(f"Interakcja: {df['interaction_type'].value_counts().to_string()}")
print(f"Special case: {df['special_case'].value_counts().to_string()}")
df[["key", "text", "speaker_id", "video_type", "interaction_type", "special_case"]].head()

Unikalni migacze: 151
Wiersze bez speaker_id: 0
Interakcja: interaction_type
dialog     25602
monolog    14215
Special case: special_case
-           39539
one_hand      278


,key,text,speaker_id,video_type,interaction_type,special_case
0,001_000_M,There is a frog in a glass jar.,F_46-60_001,monolog;obraz;komiks;Frog Story,monolog,-
1,001_001_M,The dog looks inside.,F_46-60_001,monolog;obraz;komiks;Frog Story,monolog,-
2,001_002_M,The child lies on the bed and supports his hea...,F_46-60_001,monolog;obraz;komiks;Frog Story,monolog,-
3,001_003_M,"I said, the dog looks down at the frog sitting...",F_46-60_001,monolog;obraz;komiks;Frog Story,monolog,-
4,001_004_M,"In the second picture, the girl is sleeping un...",F_46-60_001,monolog;obraz;komiks;Frog Story,monolog,-


In [12]:
fig = px.histogram(
    df,
    x="word_count",
    nbins=50,
    title="Rozkład liczby słów w tekstach",
    labels={"word_count": "Liczba słów"},
)
fig.update_yaxes(title_text="Liczba tekstów")
fig.show()

In [13]:
import unicodedata

# usuwa znaki diakrytyczne, np. "café" -> "cafe Przemyśl -> Przemysl"
def to_ascii(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")


df["clean"] = df["text"].str.lower()
# rozwijanie skrótów, np. "don't" -> "do not"
df["clean"] = df["clean"].apply(contractions.fix)
df["clean"] = df["clean"].apply(to_ascii)
df["clean"] = df["clean"].str.replace(r'[^a-z0-9\s]', '', regex=True) # może być to strict

df["clean_word_count"] = df["clean"].str.split().str.len()
fig = px.histogram(
    df,
    x="clean_word_count",
    barmode="overlay",
    title="Liczba słów po standaryzacji",
    labels={"clean_word_count": "Liczba słów"},
)
fig.update_yaxes(title_text="Liczba tekstów")
fig.show()

In [14]:
changed = df[df["word_count"] != df["clean_word_count"]]
print(f"Tekstów, które się zmieniły: {len(changed)}")
changed[["text", "clean"]].head(10)

Tekstów, które się zmieniły: 5012


,text,clean
9,She doesn't.,she does not
17,It doesn't.,it does not
22,It doesn't.,it does not
34,Don't.,do not
42,It doesn't.,it does not
61,What's next?,what is next
66,It doesn't.,it does not
74,It doesn't.,it does not
100,"It's there, it's there, you can see it through...",it is there it is there you can see it through...
104,It didn't work out.,it did not work out


#### Bigramy i trigramy

> **Sekcja eksploracyjna** — nie wpływa na pipeline filtracji.

In [15]:
from collections import Counter

bigrams = Counter()
for text in df["clean"]:
    words = text.split()
    for i in range(len(words) - 1):
        bigrams[(words[i], words[i + 1])] += 1

for (w1, w2), count in bigrams.most_common(30):
    print(f"{w1} {w2}: {count}")

it is: 2577
in the: 1914
the boy: 1714
there is: 1601
on the: 1519
is a: 1517
of the: 1187
do not: 1165
and the: 1161
the cat: 999
not allowed: 974
to the: 932
is not: 800
does not: 706
i am: 699
at the: 673
the dog: 670
the clock: 663
i have: 648
i do: 643
that is: 618
with a: 612
you can: 604
in a: 513
you have: 511
not know: 501
on a: 475
the policeman: 453
allowed to: 446
have to: 441


In [16]:
# Trigramy
trigrams = Counter()
for text in df["clean"]:
    words = text.split()
    for i in range(len(words) - 2):
        trigrams[(words[i], words[i+1], words[i+2])] += 1

for (w1, w2, w3), count in trigrams.most_common(30):
    print(f"{count:5d}  {w1} {w2} {w3}")

  917  there is a
  621  i do not
  555  of the clock
  533  is not allowed
  451  it is not
  417  do not know
  403  not allowed to
  352  it does not
  324  you have to
  302  it is a
  257  there is no
  249  is a ban
  194  out of the
  169  does not fit
  168  a ban on
  164  and the boy
  163  that it is
  157  on his head
  155  that there is
  147  to be careful
  147  are not allowed
  144  i have a
  138  have to be
  135  the alarm clock
  130  and it is
  129  in the morning
  127  the boy is
  124  go to the
  124  and the dog
  121  and there is


### Tagowanie pozycji i NER

In [17]:
import spacy 

nlp = spacy.load("en_core_web_sm")

In [18]:
from tqdm import tqdm

tqdm.pandas()

def analyze_nlp(text):
    doc = nlp(text)
    pos = [(token.lemma_.lower(), token.pos_) for token in doc]
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    return {"pos": pos, "ents": ents}

df["nlp"] = df["clean"].progress_apply(analyze_nlp)

df[["clean", "nlp"]].head(5)

100%|██████████| 39817/39817 [01:03<00:00, 629.77it/s]


,clean,nlp
0,there is a frog in a glass jar,"{'pos': [('there', 'PRON'), ('be', 'VERB'), ('..."
1,the dog looks inside,"{'pos': [('the', 'DET'), ('dog', 'NOUN'), ('lo..."
2,the child lies on the bed and supports his hea...,"{'pos': [('the', 'DET'), ('child', 'NOUN'), ('..."
3,i said the dog looks down at the frog sitting ...,"{'pos': [('i', 'PRON'), ('say', 'VERB'), ('the..."
4,in the second picture the girl is sleeping und...,"{'pos': [('in', 'ADP'), ('the', 'DET'), ('seco..."


### Lematyzacja

In [19]:
def lemmatize(text):
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

df["lemmatized"] = df["clean"].progress_apply(lemmatize)
df[["clean", "lemmatized"]].head(10)

100%|██████████| 39817/39817 [01:01<00:00, 649.17it/s]


,clean,lemmatized
0,there is a frog in a glass jar,there be a frog in a glass jar
1,the dog looks inside,the dog look inside
2,the child lies on the bed and supports his hea...,the child lie on the bed and support his head ...
3,i said the dog looks down at the frog sitting ...,I say the dog look down at the frog sit in the...
4,in the second picture the girl is sleeping und...,in the second picture the girl be sleep under ...
5,the frog is gone,the frog be go
6,she escaped,she escape
7,she disappeared,she disappear
8,the dog calls the frog out the window,the dog call the frog out the window
9,she does not,she do not


#### Usuwanie powszechnych angielskich słów

> **Sekcja eksploracyjna** — kolumny `no_stop` / `no_stop_wc` nie wpływają na pipeline filtracji. Statystyki poglądowe.

In [20]:
remove_words = {
    # przedimki
    "a", "an", "the",
    # czasowniki posiłkowe
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had",
    # zaimki dzierżawcze / wskazujące
    "it", "its", "my", "his", "her", "our", "their",
    "this", "that", "these", "those",
    # przyimki / spójniki
    "and", "or", "but", "to", "of", "in", "on", "at",
    "for", "with", "from", "by", "into", "out",
    "there", "one",
}
# {"a", "an", "the", "is", "am", "are", "was", "were", 
#                 "be", "been", "being", "do", "does", "did", 
#                 "have", "has", "had", "it", "its"}

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in remove_words])

df["no_stop"] = df["clean"].apply(remove_stopwords)

df["no_stop_wc"] = df["no_stop"].str.split().str.len()
reduction = (1 - df["no_stop_wc"].sum() / df["clean_word_count"].sum()) * 100
print(f"Redukcja słów: {reduction:.1f}%")

df[["clean", "no_stop"]].head(10)

Redukcja słów: 38.3%


,clean,no_stop
0,there is a frog in a glass jar,frog glass jar
1,the dog looks inside,dog looks inside
2,the child lies on the bed and supports his hea...,child lies bed supports head hands
3,i said the dog looks down at the frog sitting ...,i said dog looks down frog sitting jar
4,in the second picture the girl is sleeping und...,second picture girl sleeping under quilt
5,the frog is gone,frog gone
6,she escaped,she escaped
7,she disappeared,she disappeared
8,the dog calls the frog out the window,dog calls frog window
9,she does not,she not


In [21]:
fig = px.histogram(
    df, x=["clean_word_count", "no_stop_wc"], barmode="overlay",
    title="Liczba słów: przed vs po usunięciu stop-words",
    labels={"value": "Liczba słów", "variable": "Etap"}
)
fig.update_yaxes(title_text="Liczba tekstów")
fig.show()


In [22]:
print(f"Puste: {(df['no_stop_wc'] == 0).sum()}")
print(f"1 słowo: {(df['no_stop_wc'] == 1).sum()}")


Puste: 229
1 słowo: 6894


### Analiza częstotliwości występowania słów

In [23]:
word_freq = Counter()

for text in df["lemmatized"]:
    word_freq.update(text.split())

print("Unikalne słowa:", len(word_freq))

freq_df = pd.DataFrame(word_freq.most_common(50), columns=["word", "count"])
fig = px.bar(freq_df, x="word", y="count", title="50 najczęstszych słów (po lematyzacji)")
fig.update_xaxes(title_text="Słowo")
fig.update_yaxes(title_text="Liczba wystąpień")
fig.show()

Unikalne słowa: 4660


In [24]:
ent_counter = Counter()
for row in df["nlp"]:
    for text, label in row["ents"]:
        ent_counter[label] += 1

ent_counter.most_common()

[('CARDINAL', 2215),
 ('DATE', 2037),
 ('ORDINAL', 1195),
 ('PERSON', 936),
 ('TIME', 623),
 ('GPE', 202),
 ('ORG', 200),
 ('NORP', 166),
 ('LANGUAGE', 29),
 ('PERCENT', 26),
 ('QUANTITY', 21),
 ('FAC', 9),
 ('MONEY', 5),
 ('LOC', 3),
 ('LAW', 1)]

### Maskowanie rzadkich słów

> Kolumna `masked` służy **wyłącznie analityce** (obliczanie `mask_ratio`, decyzje filtracyjne).  
> **Target treningowy = kolumna `text`** (oryginalny, niezmieniony tekst).

In [25]:
THRESHOLD = 3
rare_words = {word for word, count in word_freq.items() if count <= THRESHOLD}
print(f"Rzadkie słowa (≤{THRESHOLD} wystąpień): {len(rare_words)} / {len(word_freq)}")


Rzadkie słowa (≤3 wystąpień): 2540 / 4660


In [26]:
def mask_rare(row):
    words = row["lemmatized"].split()
    nlp_data = row["nlp"]
    
    # Mapowanie: słowo -> etykieta NER
    ner_map = {}
    for ent_text, ent_label in nlp_data["ents"]:
        for w in ent_text.lower().split():
            ner_map[w] = ent_label
    
    # Mapowanie: słowo -> tag POS
    pos_map = {w.lower(): pos for w, pos in nlp_data["pos"]}
    
    result = []
    for w in words:
        if w not in rare_words:
            result.append(w)
        elif w in ner_map:
            result.append(f"<{ner_map[w]}>")
        elif w in pos_map:
            result.append(f"<{pos_map[w]}>")
        else:
            result.append("<UNK>")
    return " ".join(result)

df["masked"] = df.apply(mask_rare, axis=1)

# Ile zdań ma maski?
has_mask = df["masked"].str.contains("<").sum()
print(f"Zdania z maskami: {has_mask} / {len(df)}")

df[df["masked"].str.contains("<")][["lemmatized", "masked"]].head(10)


Zdania z maskami: 3387 / 39817


,lemmatized,masked
38,dog barking on the street,dog <NOUN> on the street
49,an owl fly overhead,an owl fly <ADV>
92,I know that mouse squirrel bird climb the tree,I know that mouse <VERB> bird climb the tree
126,on monday morning I get a job in social welfare,on monday morning I get a job in <ADJ> <NOUN>
153,we just talk look and write down the account t...,we just talk look and write down the <NOUN> th...
215,other freebie,other <NOUN>
220,the film super erotic,the film super <ADJ>
230,then from eighteen thirty to seven thirty for ...,then from eighteen thirty to seven thirty for ...
285,the 19th be free,the <DATE> be free
313,concentrate,<VERB>


#### Filtracja słów o niskiej częstotliwości występowania

In [27]:
df["mask_ratio"] = df["masked"].apply(
    lambda x: x.count("<") / max(len(x.split()), 1)
)
df["final_wc"] = df["masked"].str.split().str.len()

drop_empty = (df["final_wc"] == 0).sum()
drop_single = (df["final_wc"] == 1).sum()
drop_over_masked = (df["mask_ratio"] > 0.3).sum()

print(f"Puste: {drop_empty}")
print(f"1 słowo: {drop_single}")
print(f">30% masek: {drop_over_masked}")
print(f"Łącznie do odrzucenia: {drop_empty + drop_single + drop_over_masked}")
print(f"Zostaje: {len(df) - drop_empty - drop_single - drop_over_masked}")


Puste: 1
1 słowo: 5388
>30% masek: 533
Łącznie do odrzucenia: 5922
Zostaje: 33895


## Walidacja jakości danych

### Duplikaty

In [28]:
dupes = df["lemmatized"].duplicated()
print(f"Duplikaty: {dupes.sum()} / {len(df)} ({dupes.sum()/len(df)*100:.1f}%)")
df[df["lemmatized"].duplicated(keep=False)].sort_values("lemmatized")[["key", "text", "lemmatized"]].head(20)

Duplikaty: 9366 / 39817 (23.5%)


,key,text,lemmatized
15512,264_003_M,"""- Well?""",well
28421,527_035_M,"""- Well...""",well
35929,663_019_R,4 o'clock.,4 of the clock
33014,618_008_R,4 o'clock.,4 of the clock
37656,685_009_L,4 o'clock.,4 of the clock
1326,022_024_R,4 o'clock.,4 of the clock
37659,685_010_R,4 o'clock.,4 of the clock
23285,426_070_L,4 o'clock...,4 of the clock
37670,685_016_L,4 o'clock.,4 of the clock
32968,616_073_L,4 o'clock.,4 of the clock


### Pokrycie słownictwa

In [29]:
total_words = sum(word_freq.values())
cumsum = 0
for i, (word, count) in enumerate(word_freq.most_common(), 1):
    cumsum += count
    if cumsum / total_words >= 0.5:
        print(f"50% pokrycia: top {i} słów")
        break

cumsum = 0
for i, (word, count) in enumerate(word_freq.most_common(), 1):
    cumsum += count
    if cumsum / total_words >= 0.9:
        print(f"90% pokrycia: top {i} słów")
        break

cum_df = pd.DataFrame(word_freq.most_common(), columns=["word", "count"])
cum_df["cumulative"] = cum_df["count"].cumsum() / total_words * 100
fig = px.line(cum_df, y="cumulative", title="Kumulacyjne pokrycie słownictwa",
              labels={"index": "Liczba unikalnych słów", "cumulative": "% pokrycia korpusu"})
fig.show()

50% pokrycia: top 35 słów
90% pokrycia: top 535 słów


### Rozkład długości po pełnym pipeline

In [30]:
df["final_words"] = df["masked"].str.split().str.len()
fig = px.histogram(df, x="final_words", nbins=50, 
                   title="Rozkład długości po pełnym pipeline",
                   labels={"final_words": "Liczba tokenów"})
fig.update_yaxes(title_text="Liczba tekstów")
fig.show()

print(df["final_words"].describe())
print(f"\n95 percentyl: {df['final_words'].quantile(0.95)}")
print(f"99 percentyl: {df['final_words'].quantile(0.99)}")

count    39817.000000
mean         7.304393
std          6.103329
min          0.000000
25%          3.000000
50%          6.000000
75%         10.000000
max         58.000000
Name: final_words, dtype: float64

95 percentyl: 19.0
99 percentyl: 29.0


### Ręczna inspekcja próbek

In [31]:
import random
random.seed(42)

print("=== Krótkie (2-3 słowa) ===")
short = df[(df["final_words"] >= 2) & (df["final_words"] <= 3)].sample(5)
for _, r in short.iterrows():
    print(f"  {r['text']}  ->  {r['masked']}")

print("\n=== Średnie (6-8 słów) ===")
mid = df[(df["final_words"] >= 6) & (df["final_words"] <= 8)].sample(5)
for _, r in mid.iterrows():
    print(f"  {r['text']}  ->  {r['masked']}")

print("\n=== Długie (>15 słów) ===")
long_df = df[df["final_words"] > 15].sample(5)
for _, r in long_df.iterrows():
    print(f"  {r['text']}  ->  {r['masked']}")

print("\n=== Z maskami ===")
masked_samples = df[df["masked"].str.contains("<")].sample(5)
for _, r in masked_samples.iterrows():
    print(f"  {r['text']}  ->  {r['masked']}")

=== Krótkie (2-3 słowa) ===
  The boy confirms.  ->  the boy confirm
  The sign posted.  ->  the sign post
  He mused.  ->  he <VERB>
  Triangular sign.  ->  triangular sign
  To no avail.  ->  to no avail

=== Średnie (6-8 słów) ===
  But it is strange and opens.  ->  but it be strange and open
  I see them kissing, I get nervous.  ->  I see they kiss I get nervous
  He bought a ticket, no car.  ->  he buy a ticket no car
  It's bad, so no bathing is allowed.  ->  it be bad so no bathing be allow
  And the rest continued to party.  ->  and the rest continue to party

=== Długie (>15 słów) ===
  He walks quickly into the house, opens the door and sees a child playing a piece of fence with chopsticks like a dulcimer.  ->  he walk quickly into the house open the door and see a child play a piece of fence with <NOUN> like a dulcimer
  In the city it is not allowed, there is a ban in the city, but in the countryside it is different.  ->  in the city it be not allow there be a ban in the ci

### Analiza ile zdań pozostanie jeśli ograniczymy słownictwo do top-k% słów

In [38]:
import pandas as pd
import plotly.express as px
from collections import Counter

df = pd.read_csv("./data/processed_texts.csv")
# Częstość lematów z korpusu (nie mylić z długością zdania w final_wc)
word_freq = Counter()
for text in df["lemmatized"]:
    word_freq.update(str(text).split())
total_words = sum(word_freq.values())

In [33]:
cumsum = 0
ratio = 0.93
for i, (word, count) in enumerate(word_freq.most_common(), 1):
    cumsum += count
    if cumsum / total_words >= ratio:
        print(f"{int(ratio*100)}% pokrycia: top {i} słów")
        break
topk_words = {word for word, _ in word_freq.most_common(i)}  # i z pętli wyżej
survives = df["lemmatized"].apply(lambda x: all(w in topk_words for w in x.split()))
print(f"Zdania z samymi top-{int(ratio*100)}% słowami: {survives.sum()} / {len(df)}")
survives_path = f"./data/survives_top{int(ratio * 100)}pct.csv"
pd.DataFrame({"key": df["key"], "survives": survives}).to_csv(survives_path, index=False)
print(f"Zapisano: {survives_path}")

93% pokrycia: top 720 słów
Zdania z samymi top-93% słowami: 25548 / 39817
Zapisano: ./data/survives_top93pct.csv


In [34]:
top_list = word_freq.most_common(len(topk_words))
for word, count in top_list[-20:]:
    print(f"{count:4d}  {word}")

  38  number
  38  chair
  38  beer
  38  seat
  38  surface
  38  plant
  37  onto
  37  family
  37  sled
  37  fun
  37  snot
  37  height
  37  pour
  37  master
  37  remove
  37  tale
  37  rise
  37  branch
  37  die
  36  various


In [35]:
lost = df[~survives]
print(f"Tracone zdania: {len(lost)}")
print("\nPrzykłady traconych:")
for _, r in lost.sample(10, random_state=42).iterrows():
    print(f"  {r['text']}  ->  {r['lemmatized']}")


Tracone zdania: 14269

Przykłady traconych:
  It doesn't fit a regular shower, it fits more like scrubbing a horse.  ->  it do not fit a regular shower it fit more like scrub a horse
  He walked up the window sill.  ->  he walk up the window sill
  If alcohol is drunk, there is a high number of drownings.  ->  if alcohol be drunk there be a high number of drowning
  It pops out of the jar.  ->  it pop out of the jar
  I don't know what the code is.  ->  I do not know what the code be
  A chickadee, the bird put in.  ->  a chickadee the bird put in
  At this moment, three boys eating pears walk towards him, pass him, the man looks at them and remains silent.  ->  at this moment three boy eat pear walk towards he pass he the man look at they and remain silent
  He descends the ladder and pears into the basket until it is full.  ->  he descend the ladder and pear into the basket until it be full
  Green.  ->  green
  I propose free.  ->  I propose free


### Rozkład POS: zostające vs tracone zdania

In [36]:
skip_pos = {"DET", "ADP", "AUX", "PUNCT", "SPACE"}

def count_pos(mask):
    pos_counts = Counter()
    for row in df[mask]["nlp"]:
        for _, pos in row["pos"]:
            if pos not in skip_pos:
                pos_counts[pos] += 1
    return pos_counts

surv_pos = count_pos(survives)
lost_pos = count_pos(~survives)

pos_df = pd.DataFrame({
    "POS": list(surv_pos.keys()),
    "Zostaje": [surv_pos[p] for p in surv_pos],
    "Tracone": [lost_pos.get(p, 0) for p in surv_pos],
})
pos_df["% traconych"] = (pos_df["Tracone"] / (pos_df["Zostaje"] + pos_df["Tracone"]) * 100).round(1)
pos_df = pos_df.sort_values("% traconych", ascending=False)

fig = px.bar(pos_df, x="POS", y=["Zostaje", "Tracone"], barmode="group",
             title="Rozkład POS: zostające vs tracone zdania")
fig.show()

pos_df

,POS,Zostaje,Tracone,% traconych
8,SCONJ,2328,2854,55.1
2,NOUN,24580,28882,54.0
6,CCONJ,5459,5990,52.3
4,ADJ,6788,6868,50.3
3,ADV,7128,6518,47.8
1,VERB,26104,23313,47.2
0,PRON,21652,17837,45.2
5,PART,5470,4134,43.0
7,PROPN,2714,1862,40.7
9,NUM,2431,1354,35.8


In [39]:
import json as _json

# Serialize nlp column as JSON (faster than ast.literal_eval on reload)
df["nlp"] = df["nlp"].apply(_json.dumps)
df.to_csv("./data/processed_texts.csv", index=False)
print(f"Zapisano {len(df)} wierszy do processed_texts.csv")


Zapisano 39817 wierszy do processed_texts.csv
